In [1]:
import pandas as pd
from pathlib import Path

In [2]:
#Load and Extract Adjusted Close
RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

TICKERS = ["NVDA", "MSFT", "JNJ", "JPM", "RTX", "V"]
BENCHMARK = "^GSPC"

def load_adj_close(ticker):
    df = pd.read_csv(RAW_DIR / f"{ticker}_raw.csv")
    df["Date"] = pd.to_datetime(df["Date"])
    df = df[["Date", "Adj Close"]].rename(columns={"Adj Close": ticker})
    return df

# Load assets
dfs = [load_adj_close(t) for t in TICKERS]

In [3]:
#Merge All Assets on Date
from functools import reduce

prices = reduce(
    lambda left, right: pd.merge(left, right, on="Date", how="inner"),
    dfs
)

prices = prices.sort_values("Date").reset_index(drop=True)
prices.head()

,Date,NVDA,MSFT,JNJ,JPM,RTX,V
0,2014-01-02,0.3738640546798706,30.831125259399414,65.215087890625,42.19771194458008,53.383018493652344,50.66695785522461
1,2014-01-03,0.3693852126598358,30.62371253967285,65.80254364013672,42.52392578125,53.57756423950195,50.70135498046875
2,2014-01-06,0.37433546781539917,29.976552963256836,66.14642333984375,42.77039337158203,53.5253791809082,50.396453857421875
3,2014-01-07,0.3804643750190735,30.208871841430664,67.55056762695312,42.27743911743164,53.867069244384766,50.78159713745117
4,2014-01-08,0.3856503367424011,29.669572830200195,67.45746612548828,42.6761589050293,53.93350601196289,50.94437026977539


In [5]:
#Coercing to correct dtype
price_cols = [c for c in prices.columns if c != "Date"]
prices[price_cols] = prices[price_cols].apply(pd.to_numeric, errors="coerce")

#Compute Daily Returns
returns = prices.copy()
returns = returns.dropna(subset=["Date"]).sort_values("Date")
returns.set_index("Date", inplace=True)

returns = returns.pct_change().dropna()
returns.head()

,NVDA,MSFT,JNJ,JPM,RTX,V
Date,,,,,,
2014-01-03,-0.011980,-0.006727,0.009008,0.007731,0.003644,0.000679
2014-01-06,0.013401,-0.021133,0.005226,0.005796,-0.000974,-0.006014
2014-01-07,0.016373,0.007750,0.021228,-0.011526,0.006384,0.007642
2014-01-08,0.013631,-0.017852,-0.001378,0.009431,0.001233,0.003205
2014-01-09,-0.037286,-0.006432,0.006053,-0.001869,0.000528,-0.001395


In [10]:
#Sanity Checks
returns.describe()
returns.isnull().sum()
returns.shape

(2766, 6)

In [11]:
#Save Clean Artifacts
prices.to_csv(PROCESSED_DIR / "prices_aligned.csv", index=False)
returns.to_csv(PROCESSED_DIR / "returns_daily.csv")